# Phase 02: Data Cleaning & Preprocessing
Prepares the raw student dataset for machine learning by applying consistency rules, feature engineering, stratified train-test splitting, and defining modular encoding/scaling pipelines.

## 1. Imports & Environment Setup
Import core libraries and custom modular preprocessing functions from `src.preprocess`.

In [1]:
import sys
sys.path.append("..")

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from src.preprocess import (
    correct_feature,
    target_isolation,
    feats_engineering,
    split_data,
    feats_categorization,
    numericals_pipeline,
    nominals_pipeline,
    ordinals_pipeline,
    preprocessor_col_transformer
)

## 2. Load Raw Dataset
Configure relative paths and load the raw dataset (`student_mental_health.csv`).

In [3]:
root = Path.cwd().parent

In [4]:
DATA_PATH = root / "data" / "origine" / "student_mental_health.csv"

In [5]:
df_clean = pd.read_csv(DATA_PATH)

## 3. Consistency Correction
Cap `Screen_Time_Hours` to `Social_Media_Hours` where social media usage exceeded total screen time.

In [6]:
df_clean = correct_feature(df_clean)

## 4. Feature Engineering
Convert weekly study hours (`Study_Hours_Per_Week`) into daily `Study_Hours` and drop the weekly feature.

In [7]:
df_clean = feats_engineering(df_clean)

## 5. Target Isolation & ID Removal
Separate predictor features `X` from target `y` (`Mental_Health_Risk`) and exclude `Student_ID`.

In [8]:
X, y = target_isolation(df_clean)

## 6. Stratified Train-Test Split
Partition data into an 80/20 train-test split ($1,600$ train, $400$ test) with stratified sampling (`random_state=42`).

In [9]:
X_train, X_test, y_train, y_test = split_data(X, y)

## 7. Verify Split Shapes
Check the dimensions of feature matrices and target vectors across train and test sets.

In [10]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1600, 15)
X_test: (400, 15)
y_train: (1600,)
y_test: (400,)


## 8. Verify Class Balance
Confirm that the target class distribution (~33% per class) is preserved across both splits.

In [11]:
train_distribution = pd.DataFrame({
    "Count": y_train.value_counts(),
    "Percentage": y_train.value_counts(normalize=True) * 100
})

test_distribution = pd.DataFrame({
    "Count": y_test.value_counts(),
    "Percentage": y_test.value_counts(normalize=True) * 100
})

print("Training distribution:")
display(train_distribution)

print("Testing distribution:")
display(test_distribution)

Training distribution:


,Count,Percentage
Mental_Health_Risk,,
High,544,34.0
Medium,528,33.0
Low,528,33.0


Testing distribution:


,Count,Percentage
Mental_Health_Risk,,
High,136,34.0
Medium,132,33.0
Low,132,33.0


## 9. Feature Categorization
Group the 15 features into numerical (10), nominal (3), and ordinal (2) sets for specialized transformations.

In [12]:
numerical_features, nominal_features, ordinal_features = feats_categorization()

## 10. Numerical Feature Pipeline
Median imputation for missing values followed by standard scaling (`StandardScaler`).

In [13]:
numeric_pipeline = numericals_pipeline()

## 11. Nominal Feature Pipeline
Most-frequent imputation followed by one-hot encoding (`OneHotEncoder(handle_unknown='ignore')`).

In [14]:
nominal_pipeline = nominals_pipeline()

## 12. Ordinal Feature Pipeline
Most-frequent imputation followed by ordinal encoding (`OrdinalEncoder`) with explicit category ordering.

In [15]:
ordinal_pipeline = ordinals_pipeline()


## 13. ColumnTransformer Assembly
Combine numerical, nominal, and ordinal pipelines into a unified `ColumnTransformer`.

In [16]:
preprocessor = preprocessor_col_transformer(
    numeric_pipeline,
    numerical_features,
    nominal_pipeline,
    nominal_features,
    ordinal_pipeline,
    ordinal_features,
)

## 14. Pipeline Anti-Leakage Strategy
The `preprocessor` is defined without fitting. It will be fitted strictly within cross-validation folds in Phase 03 to prevent data leakage.

## 15. Summary & Next Steps
- **Cleaned & Split:** Inconsistency resolved, feature engineered, and data split 80/20 with stratification.
- **Pipelines Ready:** Modular `ColumnTransformer` defined and ready to integrate into Phase 03 model pipelines (Logistic Regression, Random Forest, XGBoost, SVM).